In [ ]:
# load required libraries
import sys
sys.path.append('../utils')
# data model - Reading annotated data (by annotators)
# the data model class below returns masks as well
from json_parser import CellMaskDataset, MaskDatasetFromMultiAnnotations
from json_parser import optimize_crop, crop_and_block, show_sample, enforce_one_to_one_mapping

import os
from PIL import Image
from IPython.display import display
import numpy as np
import shutil
import cv2
import pandas as pd
import pickle
import pycocotools
import random
from pycocotools import mask as coco_mask_util
from typing import List, Union, Dict, Final, Tuple, Optional

In [ ]:
MAX_IMAGE_SIDE = 4512
RESIZED_BB_IMAGE_SIZE = 4512 ###### 2720 for Mask R-CNN
RATIO_OF_IMAGES_TO_USE_FOR_SUSPENSION_CAGED_DATASETS = 0.25   ###### was 0.2
RATIO_OF_IMAGES_TO_USE_FOR_SUSPENSION_UNCAGED_DATASETS = 0.05 ###### was 0.02
RATIO_OF_IMAGES_TO_USE_FOR_ADHERED_DATASETS = 1.0

CLASS_NAMES_TO_CLASS_IDS_MAP =  {'soma': 3,
                                 'cytoplasm': 4,  'cell-adhered': 4,
                                  # 'cage': 3, 'cages': 3, 
                                 'bead': 2, 'Bead': 2,
                                 'cell': 1, 'Cell': 1, 'dead-cell': 1, 'dying/dead cells': 1}

NUM_CLASSES = len(set(CLASS_NAMES_TO_CLASS_IDS_MAP.values()))

CLASS_NAMES_TO_USE_FOR_CROP_OVERLAPS = ['cell', 'cell-adhered', 'soma',] # 'cage']

NUM_RANDOM_IMAGES_TO_CONSIDER_IN_Z_STACK_SET: int = 2
# the minimum object mask area to keep the object in the data
MIN_MASK_AREA = 16

CHILD_PARENT_CLASS_MAP = None
# the limit on the larger and smaller sides of the input to Mask R-CNN model
# used for preparing un-cropped training images (whole images)
MASK_RCNN_INPUT_WIDTH = 672 ###### was 1024
MASK_RCNN_INPUT_HEIGHT = 672 ###### was 800

DATASET_PATHS = [# new datasets 31
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/231212_imr90_multichannel_overlay', # no z-stack
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/240213_imr90_multichannel_overlay', # no z-stack
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_jurkat_10x_caged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_jurkat_10x_uncaged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_k562_10x_caged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_k562_10x_uncaged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_nk92_10x_caged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_nk92_10x_uncaged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240425_nk92_10x_caged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240425_nk92_10x_uncaged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_hela-suspension_10x_caged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_hela-suspension_10x_uncaged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240314_imr90-suspension_10x_caged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240314_imr90-suspension_10x_uncaged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240307_pbmc-beads_10x_uncaged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240305_pbmc-nobeads_10x_caged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240305_pbmc-nobeads_10x_uncaged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240306_mousepbmc-beads_10x_caged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240306_mousepbmc-beads_10x_uncaged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240306_mousepbmc-nobeads_10x_caged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240306_mousepbmc-nobeads_10x_uncaged',
    '/home/cellareye/Cellanome/Data/20240422_neuron-adhered_10x_uncaged',
    '/home/cellareye/Cellanome/Data/20240509_Hs675Tfibroblasts_10x_caged',
    '/home/cellareye/Cellanome/Data/20240509_hela-adhered_10x_caged',
    '/home/cellareye/Cellanome/Data/20240515_DC-adhered_10x_caged', # no z-stack
    '/home/cellareye/Cellanome/Data/20240516_DC-adhered_10x_caged', # no z-stack
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240624_mc38_10x_caged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240624_mc38_10x_uncaged',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240625_mc38_10x_caged',
    '/home/cellareye/Cellanome/Data/20240703_neuron-adhered_10x_caged',
    '/home/cellareye/Cellanome/Data/20240704_neuron-adhered_10x_caged',
    '/home/cellareye/Cellanome/Data/20240816_tall104_10x_caged_at_4x', # no z-stack
    '/home/cellareye/Cellanome/Data/20240905_raji_10x_caged_at_4x',
    '/home/cellareye/Cellanome/Data/20240905_u87-adhered_10x_caged',
    '/home/cellareye/Cellanome/Data/20240924_enteric-glia-adhered_10x_uncaged',
    '/home/cellareye/Cellanome/Data/20241003_Hs675Tfibroblasts-suspension-beads_10x_uncaged',
    # old datasets (9 sets)
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/231115_k562_cells',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/d1_ix81',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/normalised_12212022_tregs_beads_cages_bb2_bb2',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/normalised_cytokine_nk_10312022_bb2',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/normalised_cytokine_nk_10312022_ix81',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/normalised_cytokine_pbmc_10312022_ix81',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/normalised_surface_jurkat_10312022_bb2',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/normalised_surface_nk_jurkat_10312022_ix81',
    '/media/cellareye/SSD/Data/Cellanome/Segmentation/normalised_surface_pbmc_10312022_ix81'
]

SUSPENSION_DATASETS = [
    '20240228_jurkat_10x_caged',
    '20240228_jurkat_10x_uncaged',
    '20240228_k562_10x_caged',
    '20240228_k562_10x_uncaged',
    '20240228_nk92_10x_caged',
    '20240228_nk92_10x_uncaged',
    '20240425_nk92_10x_caged',
    '20240425_nk92_10x_uncaged',
    '20240228_hela-suspension_10x_caged',
    '20240228_hela-suspension_10x_uncaged',
    '20240314_imr90-suspension_10x_caged',
    '20240314_imr90-suspension_10x_uncaged',
    '20240307_pbmc-beads_10x_uncaged',
    '20240305_pbmc-nobeads_10x_caged',
    '20240305_pbmc-nobeads_10x_uncaged',
    '20240306_mousepbmc-beads_10x_caged',
    '20240306_mousepbmc-beads_10x_uncaged',
    '20240306_mousepbmc-nobeads_10x_caged',
    '20240306_mousepbmc-nobeads_10x_uncaged',
    '20240816_tall104_10x_caged_at_4x',
    '20240905_raji_10x_caged_at_4x',
    '20241003_Hs675Tfibroblasts-suspension-beads_10x_uncaged',
    '231115_k562_cells',
    'd1_ix81',
    'normalised_12212022_tregs_beads_cages_bb2_bb2',
    'normalised_cytokine_nk_10312022_bb2',
    'normalised_cytokine_nk_10312022_ix81',
    'normalised_cytokine_pbmc_10312022_ix81',
    'normalised_surface_jurkat_10312022_bb2',
    'normalised_surface_nk_jurkat_10312022_ix81',
    'normalised_surface_pbmc_10312022_ix81'
]

In [ ]:
def create_dataset_classes(dataset_path: str, 
                           class_names_to_class_ids_map: Dict[str, int], 
                           percentage_to_expand_bbox_boundaries: float = 0.0, 
                           max_images_to_consider_for_each_annotation: int = 1,
                           only_use_best_focus_image: bool = False, 
                           max_larger_side: int = MAX_IMAGE_SIDE, 
                           max_smaller_side: int = MAX_IMAGE_SIDE, 
                          ):
    
    images_path: str = dataset_path
    annotations_path: str = os.path.join(dataset_path, 'annotations')

    annotations_images_map: pd.DataFrame = pd.read_csv(os.path.join(dataset_path, 'annotation_images_mapping.csv'))

    test_files: List[str] = []
    train_files: List[str] = []
    
    with open(os.path.join(dataset_path, 'test.txt')) as file:
        filenames: List[str] = file.readlines()
        filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
        test_files += filenames

    with open(os.path.join(dataset_path, 'train.txt')) as file:
        filenames: List[str] = file.readlines()
        filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
        train_files += filenames

    if os.path.exists(os.path.join(dataset_path, 'classes.txt')):
        # open the classes.txt if exists 
        class_names: List[str] = []
        with open(os.path.join(dataset_path, 'classes.txt')) as file:
            class_name: List[str] = file.readlines()
            class_name = [f.replace('\n', '') for f in class_name if len(f) > 0]
            class_names += class_name
        
        # print(f"[INFO] List of class names in dataset classes.txt file but not defined in class_names_to_class_ids_map:"
        #       f" {[s for s in class_names if s not in class_names_to_class_ids_map]}")
        # if len([s for s in class_names if s not in class_names_to_class_ids_map]) > 0:
            # print("[WARN]: There are some classes in the dataset not covered in class_names_to_class_ids_map!")
    
        # print(f"[INFO] List of class names defined in class_names_to_class_ids_map but not in dataset classes.txt file:"
        #       f" {[s for s in class_names_to_class_ids_map if s not in class_names]}")
    
    
    columns: List[str] = list(annotations_images_map.columns)
    image_columns: List[str] = [column_name for column_name in columns if 'white_dz' in column_name.lower()]

    num_images_to_consider_for_each_annotation: int = max_images_to_consider_for_each_annotation
    
    if len(image_columns) > 1: 
        if only_use_best_focus_image:
            image_columns = [column_name for column_name in columns if 'white_dz0' in column_name.lower()]
            num_images_to_consider_for_each_annotation = 1
            # print('[INFO]: This is a z-stack dataset, but the best focus distance image is selected because of the passed configuration.')
        else:
            pass
            # print(f"[INFO]: This is a z-stack dataset. {max_images_to_consider_for_each_annotation} randomly chosen images will be considered.")
        
    if len(image_columns) == 0:
        # this is not a focus sweep dataset, get the BF image
        for column_name in columns:
            if 'bf' in column_name.lower() or 'white' in column_name.lower() or 'images' in column_name.lower():
                image_columns = [column_name]
                num_images_to_consider_for_each_annotation = 1
                # print(f"[INFO]: This is NOT a z-stack dataset. {column_name} channel will be considered as the brightfield channel.")
                break

    if len(image_columns) == 0:
        print('[ERROR]: No brightfield image folder could be extracted from annotation_images_mapping.csv file for the dataset.')
    
    train_map_dict: Dict[str, List[str]] = {}
    test_map_dict: Dict[str, List[str]] = {}
    for _, row in annotations_images_map.iterrows():
        annotations_filename = row['annotation_json']
        name = '.'.join(annotations_filename.strip().split('.')[:-1])
        if name in test_files:
            test_map_dict[annotations_filename] = list(row[image_columns])
        else:
            train_map_dict[annotations_filename] = list(row[image_columns])


    # dataset classes
    train_dataset = CellMaskDataset(images_path=images_path, annotations_path=annotations_path, 
                                    annotations=train_map_dict,
                                    max_images_to_consider_for_each_annotation=num_images_to_consider_for_each_annotation, 
                                    labels_of_interest=list(class_names_to_class_ids_map.keys()), 
                                    percentage_to_expand_bbox_boundaries = percentage_to_expand_bbox_boundaries, 
                                    color_depth=8, 
                                    min_object_diameter = 6.0,
                                    scale_factor_dict={(2000, 1600): 1.11111111},  # for ix-81 microscope images in old datasets
                                    max_larger_side = max_larger_side, max_smaller_side = max_smaller_side,
                                    normalize=False, class_names_to_ids_map=class_names_to_class_ids_map)

    
    test_dataset = CellMaskDataset(images_path=images_path, annotations_path=annotations_path, 
                                   annotations=test_map_dict,
                                   max_images_to_consider_for_each_annotation=num_images_to_consider_for_each_annotation,
                                   labels_of_interest=list(class_names_to_class_ids_map.keys()), 
                                   percentage_to_expand_bbox_boundaries = percentage_to_expand_bbox_boundaries, 
                                   color_depth=8, 
                                   min_object_diameter = 6.0,
                                   scale_factor_dict={(2000, 1600): 1.11111111}, # for ix-81 microscope images in old datasets
                                   max_larger_side = max_larger_side, max_smaller_side = max_smaller_side,
                                   normalize=False, class_names_to_ids_map=class_names_to_class_ids_map)

    return train_dataset, test_dataset

def prepare_data(dataset, 
                 image_folder: str,
                 mask_folder: str,
                 train=True,
                 save_masks_in_coco_rle_format=True, 
                 overlap_in_x=176, 
                 overlap_in_y=160, 
                 child_parent_map=None, 
                 ratio_of_samples_to_use: float = 1.0):
    
    if train:
        set_desc = 'training'
    else:
        set_desc = 'testing'

    # keep all the labels in the model label map
    class_ids_of_interest = list(set(CLASS_NAMES_TO_CLASS_IDS_MAP.values()))
    
    num_images = 0
    num_annotations = 0
    num_samples = len(dataset)

    idxs_to_use = np.array([i for i in range(num_samples)])
    if ratio_of_samples_to_use < 1.0:
        # always use the same seed
        random.seed(7)
        random.shuffle(idxs_to_use)
        idxs_to_use = idxs_to_use[:int(num_samples * ratio_of_samples_to_use)]

    print(f"[INFO] Using {len(idxs_to_use)} samples for cropping {set_desc} data!")
    # read the image and the annotations, then parse each
    for idx in idxs_to_use:

        sample = dataset[idx]
        
        if len(sample['annotations']) == 0:
            continue
        
        # image size
        image_height, image_width = sample["image"].shape[:2]
        
        if (image_width, image_height) not in [(1800, 1440), (RESIZED_BB_IMAGE_SIZE, RESIZED_BB_IMAGE_SIZE)]:
            print('[WARNING: Incorrect image resolution! Skipping the image ...')
            continue
            
        # special override of the crop overlaps for ix-81 microscope images
        # (early datasets have some mixture of them)
        if image_height == 1440 and image_width == 1800:
            overlap_in_x = 248
            overlap_in_y = 608 ###### was 160
            
        # the step size for the starting point of each crop in x and y dimension
        crop_start_step_x = MASK_RCNN_INPUT_WIDTH - overlap_in_x
        crop_start_step_y = MASK_RCNN_INPUT_HEIGHT - overlap_in_y


        crop_count = 0
        # overlapping crops
        for x_start in range(0, image_width - overlap_in_x, crop_start_step_x):    
            for y_start in range(0, image_height - overlap_in_y, crop_start_step_y):
                # crop coordinates
                xc_tl = x_start
                yc_tl = y_start
                xc_br = x_start + MASK_RCNN_INPUT_WIDTH
                yc_br = y_start + MASK_RCNN_INPUT_HEIGHT
                # make sure we always crop the image with the given size
                # if we get to the boundaries, extend the crop
                # size inside the image to always get the same size crop
                # this is not really needed, but help with capturing more
                # annotations toward the low/right parts of the image
                if xc_br > image_width:
                    xc_br = image_width 
                    xc_tl = max(0, xc_br - MASK_RCNN_INPUT_WIDTH)
                if yc_br > image_height:
                    yc_br = image_height
                    yc_tl = max(0, yc_br - MASK_RCNN_INPUT_HEIGHT)

                crop_coords = [xc_tl, yc_tl, xc_br, yc_br]

                # optimize the crop
                crop_coords = optimize_crop(sample['annotations'], crop_coords, 
                                            int(overlap_in_x / 3), int(overlap_in_y / 3), 
                                            image_width, image_height, class_ids_of_interest, 
                                            sample['masks'])

                # crop and block the image
                cropped_sample =  crop_and_block(sample, crop_coords, labels_of_interest=class_ids_of_interest, 
                                                 keep_area_threshold=0.33)
                
                # skip empty crops
                if len(cropped_sample['annotations']) < 1:
                    continue
                
                # enforce the one-to-one mapping on the cropped image to ensure 
                # removing cases where the mapping is broken by the cropping process above
                
                if child_parent_map is not None:
                    cropped_sample = enforce_one_to_one_mapping(data_sample=cropped_sample, 
                                                                child_parent_map=child_parent_map)

                crop_height, crop_width = cropped_sample["image"].shape[:2]
                
                # cropped image and annotations name, for each image/annotation name use _ crop_count
                img_name = ".".join(cropped_sample["name"].strip().split('.')[:-1])
                # image jpg file
                crp_img_name = img_name + '_crp_' + str(crop_count) + '.jpg'
                
                if save_masks_in_coco_rle_format:
                    # save the masks and annotations as a dictionary (pickle file)
                    # as in the following
                    record = {}
                    record['annotations']: list[dict] = []
                    for obj_id, current_mask in enumerate(cropped_sample["masks"]):
                    
                        # check the area first, make sure to include only large enough objects
                        if current_mask.sum() < MIN_MASK_AREA:
                            continue
                    
                        # object's bounding box
                        box_xtl, box_ytl, box_xbr, box_ybr = \
                        cropped_sample['annotations'].loc[obj_id, ['xtl', 'ytl', 'xbr', 'ybr']].values
                        
                        # object's label
                        label = int(cropped_sample['annotations'].loc[obj_id, 'label'])
                        
                        # we do not expand the mask to the crop's resolution and
                        # only save the mask within the objects bounding box to save space
                        num_annotations += 1
                  
                        annots: dict = {'bbox': [box_xtl, box_ytl, box_xbr, box_ybr],
                                        'category_id' : label,
                                        'segmentation': coco_mask_util.encode(np.asarray(current_mask, 
                                                                                         order="F")),
                                       } 
                        record['annotations'].append(annots)
                    
                    if len(record['annotations']) == 0:
                        # skip this sample
                        continue
                    
                    # save the record dictionary as a pickle file
                    crp_mask_name = img_name + '_crp_' + str(crop_count) + '.pkl'
                    
                    filehandler = open(os.path.join(OUTPUT_BASE_PATH, mask_folder, crp_mask_name), 'wb')
                    pickle.dump(record, filehandler)
                    filehandler.close()
                    
                    
                else:
                    # save the mask annotations in our internal format as below:
                    # the masks are saved as an m x crop_height x crop_width array 
                    # (m masks with the same resolution as the cropped image)
                    # and values as np.uint16 
                    # the i-th object in cropped_sample['annotations'] will have 
                    # mask values eqaul to (i + 1) in the first possible array index j
                    # 0 <= j < m where j is chosen such that this mask will have any overlap
                    # with the objects saved on this slice (j)
                    # in the case of having overlapping objects, one of the objects will
                    # be reported in the next/different array index where it does not have 
                    # any overlap with objects in that mask

                    masks: List[np.array] = [np.zeros((crop_height, crop_width), np.uint16)]
                    labels: np.array = np.zeros((len(cropped_sample["masks"]),),  np.uint8) 
                    
                    for obj_id, current_mask in enumerate(cropped_sample["masks"]):
                    
                        # check the area first, make sure to include only large enough objects
                        if current_mask.sum() < MIN_MASK_AREA:
                            continue
                    
                        box_xtl, box_ytl, box_xbr, box_ybr = \
                        cropped_sample['annotations'].loc[obj_id, ['xtl', 'ytl', 'xbr', 'ybr']].values
                        # expand the object mask to cover the full crop resolution
                        full_res_mask: np.array = np.zeros((crop_height, crop_width), np.uint8)
                        full_res_mask[box_ytl:box_ybr, box_xtl:box_xbr] = 1
                    
                        num_annotations += 1
                        # find the first index with no overlap for the object
                        available_array_index = -1
                        for array_index in range(len(masks)):
                            if masks[array_index][full_res_mask > 0].sum() == 0:
                                available_array_index = array_index
                                break

                        if available_array_index < 0:
                            # add a new mask
                            masks.append(np.zeros((crop_height, crop_width), np.uint16))
                            available_array_index = len(masks) - 1

                        masks[available_array_index][full_res_mask > 0] = obj_id + 1
                        labels[obj_id] = int(cropped_sample['annotations'].loc[obj_id, 'label'])

                    
                    if len(masks) == 0 or masks[0].sum() == 0:
                        # skip this cropped sample if no large enough mask was found
                        continue
                
                    # save the annotations (masks) for this image, 
                    # annotation npz file
                    crp_mask_name = img_name + '_crp_' + str(crop_count) + '.npz'

                    # save the masks and labels arrays
                    np.savez(os.path.join(OUTPUT_BASE_PATH, mask_folder, crp_mask_name), 
                             saved_masks = np.array(masks), saved_labels = labels) 
                
                # save the image in jpg format
                cv2.imwrite(os.path.join(OUTPUT_BASE_PATH, image_folder, crp_img_name), cropped_sample["image"])
                
                # increament the counter for images
                num_images += 1
                # increament the crop counter for this image
                crop_count += 1


    print('[INFO] Created {} '.format(num_images) + 'images for ' + set_desc)
    print('[INFO] Created {} '.format(num_annotations) + 'objects for ' + set_desc)


######
# these are default overlap values reasonable for cells and other objects, they will be updated if needed below
# DEFAULT_X_OVERLAP = 176
# DEFAULT_Y_OVERLAP = 160
# DEFAULT_X_OVERLAP_LARGE_OBJS = 458
# DEFAULT_Y_OVERLAP_LARGE_OBJS = 416

######
# two dictionaries for looking up the required overlap in x and y in cropping given the object size of 
# interest (cage here)
# the keys are the object size range in (min, max) and the values  are the overlap values in x and y
# (the overlap is applicable if the object size fall in that range
# OVERLAP_LOOKUP = {(1, 224): (176, 160),
#                   (225, 244): (176, 320),
#                   (245, 444): (458, 320),
#                   (445, 584): (458, 416),
#                   (585, 640): (458, 480)}

DEFAULT_X_OVERLAP = 200
DEFAULT_Y_OVERLAP = 200
DEFAULT_X_OVERLAP_LARGE_OBJS = 400
DEFAULT_Y_OVERLAP_LARGE_OBJS = 400

###### 
OVERLAP_LOOKUP = {(1, 44): (32, 32),
                  (45, 174): (123, 123),
                  (175, 264): (192, 192),
                  (265, 344): (245, 245),
                  (345, 404): (288, 288),
                  (405, 454): (322, 322),
                  (455, 494): (352, 352),
                  (495, 524): (376, 376),
                  (525, 537): (397, 397)}

def get_object_size(dataset, reverse_label_map, class_names_of_interest: List[str] = ['cage']):

    object_size: Dict[str, int] = {key: -1 for key in class_names_of_interest}
    object_widths: Dict[str, List[int]] = {}
    object_heights: Dict[str, List[int]] = {}
    class_ids_to_check: Dict[str, int] = {}
    for class_name in class_names_of_interest:
        if class_name in reverse_label_map:
            class_id: int = reverse_label_map[class_name]
            object_widths[class_name] = []
            object_heights[class_name] = []
            class_ids_to_check[class_name] = class_id

    if len(class_ids_to_check) == 0:
        print(f"[ERROR] class_names_of_interest: {class_names_of_interest} and keys of reverse_label_map:", 
              f"{list(reverse_label_map.keys())} do not have any overlap")
        return object_size
    
    for idx in range(len(dataset)):
        data = dataset[idx]
        for class_name, class_id in class_ids_to_check.items():
            df = data['annotations']
            filterd_df = df[df['label']==class_id]
            heights = (filterd_df['ybr'] - filterd_df['ytl']).values
            widths = (filterd_df['xbr'] - filterd_df['xtl']).values
            object_widths[class_name] += list(widths)
            object_heights[class_name] += list(heights)

    for class_name in class_ids_to_check:
        if len(object_widths[class_name]) > 0:
            object_size[class_name] = np.mean([np.percentile(object_widths[class_name], 95), 
                                               np.percentile(object_heights[class_name], 95)])

    return object_size
            
def get_crop_overlaps(dataset):
    if ((MASK_RCNN_INPUT_WIDTH != 1024 or MASK_RCNN_INPUT_HEIGHT != 800 or RESIZED_BB_IMAGE_SIZE != 2720) and 
        (MASK_RCNN_INPUT_WIDTH != 672 or MASK_RCNN_INPUT_HEIGHT != 672 or RESIZED_BB_IMAGE_SIZE != 4512)):
            print("[ERROR] The pre-calculated crop overlaps are no longer applicable. Re-calculate them ...")
            x_overlap = -1
            y_overlap = -1
    else:
        object_sizes = get_object_size(dataset, CLASS_NAMES_TO_CLASS_IDS_MAP, CLASS_NAMES_TO_USE_FOR_CROP_OVERLAPS)

        object_size = -1
        if len([value for value in object_sizes.values()]) > 0:
            object_size = max([value for value in object_sizes.values()])
            print(f"[INFO] The object size to be used for overlap calculation: {object_size}")

        if object_size < 0:
            x_overlap = DEFAULT_X_OVERLAP
            y_overlap = DEFAULT_Y_OVERLAP
            # print(f"[WARN] No object size could be found to use for calculating the crop size!", 
            #       f"The default crop sizes  {x_overlap, y_overlap} will be used")
        else:
            found = False
            for (lb, ub), (ol_x, ol_y) in OVERLAP_LOOKUP.items():
                if lb <= object_size <= ub:
                    found = True
                    x_overlap = ol_x
                    y_overlap = ol_y
                    # print(f"[INFO] Crop overlap values successfully set to {x_overlap, y_overlap}")
                    break
            
            if not found:
                x_overlap = DEFAULT_X_OVERLAP_LARGE_OBJS
                y_overlap = DEFAULT_Y_OVERLAP_LARGE_OBJS
                # print(f"[WARN] One of the objects are too big!  Default crop overlap values of {x_overlap, y_overlap} will be used!")
    
    return x_overlap, y_overlap
            

In [ ]:
for dataset_path in DATASET_PATHS:
    
    dataset_name: str = dataset_path.strip().split('/')[-1]
    
    suspension_dataset: bool = dataset_name in SUSPENSION_DATASETS
    caged: bool = '_cage' in dataset_name

    train_dataset, test_dataset = create_dataset_classes(dataset_path=dataset_path, 
                                                         class_names_to_class_ids_map=CLASS_NAMES_TO_CLASS_IDS_MAP, 
                                                         percentage_to_expand_bbox_boundaries=0.0, 
                                                         max_images_to_consider_for_each_annotation=NUM_RANDOM_IMAGES_TO_CONSIDER_IN_Z_STACK_SET,
                                                         # if the dataset has only 1 image per annotation, this will be ignored
                                                         only_use_best_focus_image=False,
                                                         # only works for datasets with z-stacks
                                                         max_larger_side=RESIZED_BB_IMAGE_SIZE, 
                                                         max_smaller_side=RESIZED_BB_IMAGE_SIZE)
    
    print(f"[INFO] Number of training images in the dataset {dataset_name}: {len(train_dataset)}")
    print(f"[INFO] Number of testing images in the dataset {dataset_name}: {len(test_dataset)}")

    x_overlap, y_overlap = get_crop_overlaps(test_dataset)
    
    if x_overlap < 0 or y_overlap < 0:
        print(f"[ERROR] Impossible to compute overlaps! Skipping dataset {dataset_name}")
        continue

    print(f"[INFO] Crop overlap values successfully set to {x_overlap, y_overlap}")
    # create output folders
    ## make sure these folders are generated in advance
    FOLDER_NAME = dataset_name + '_cropped_for_DINOv2_' + str(NUM_CLASSES) + '_class'
    OUTPUT_BASE_PATH = os.path.join(os.getcwd(), 'data', FOLDER_NAME)
    TRAIN_IMAGE_FOLDER = 'images/train'
    TRAIN_MASK_FOLDER = 'masks/train'
    TEST_IMAGE_FOLDER = 'images/test'
    TEST_MASK_FOLDER = 'masks/test'

    if not os.path.exists(os.path.join(os.getcwd(), 'data')):
        os.mkdir(os.path.join(os.getcwd(), 'data'))
    
    if not os.path.exists(OUTPUT_BASE_PATH):
        os.mkdir(OUTPUT_BASE_PATH)
    
    if not os.path.exists(os.path.join(OUTPUT_BASE_PATH, 'images')):
        os.mkdir(os.path.join(OUTPUT_BASE_PATH, 'images'))
    
    if not os.path.exists(os.path.join(OUTPUT_BASE_PATH, 'masks')):
        os.mkdir(os.path.join(OUTPUT_BASE_PATH, 'masks'))
    
    if not os.path.exists(os.path.join(OUTPUT_BASE_PATH, TRAIN_IMAGE_FOLDER)):
        os.mkdir(os.path.join(OUTPUT_BASE_PATH, TRAIN_IMAGE_FOLDER))
    
    if not os.path.exists(os.path.join(OUTPUT_BASE_PATH, TRAIN_MASK_FOLDER)):
        os.mkdir(os.path.join(OUTPUT_BASE_PATH, TRAIN_MASK_FOLDER))
    
    if not os.path.exists(os.path.join(OUTPUT_BASE_PATH, TEST_IMAGE_FOLDER)):
        os.mkdir(os.path.join(OUTPUT_BASE_PATH, TEST_IMAGE_FOLDER))
    
    if not os.path.exists(os.path.join(OUTPUT_BASE_PATH, TEST_MASK_FOLDER)):
        os.mkdir(os.path.join(OUTPUT_BASE_PATH, TEST_MASK_FOLDER))

    if suspension_dataset:
        if caged:
            ratio_of_samples_to_use = RATIO_OF_IMAGES_TO_USE_FOR_SUSPENSION_CAGED_DATASETS
        else:
            ratio_of_samples_to_use = RATIO_OF_IMAGES_TO_USE_FOR_SUSPENSION_UNCAGED_DATASETS
    else:
        ratio_of_samples_to_use = RATIO_OF_IMAGES_TO_USE_FOR_ADHERED_DATASETS    
        
    print(f"[INFO] Using {ratio_of_samples_to_use * 100}% of samples for train/test") 
    prepare_data(train_dataset,
                 TRAIN_IMAGE_FOLDER,
                 TRAIN_MASK_FOLDER,
                 train=True, 
                 save_masks_in_coco_rle_format=True, 
                 overlap_in_x=x_overlap, overlap_in_y=y_overlap, 
                 child_parent_map=CHILD_PARENT_CLASS_MAP, 
                 ratio_of_samples_to_use=ratio_of_samples_to_use)
    
    prepare_data(test_dataset,
                 TEST_IMAGE_FOLDER,
                 TEST_MASK_FOLDER,
                 train=False, 
                 save_masks_in_coco_rle_format=True, 
                 overlap_in_x=x_overlap, overlap_in_y=y_overlap, 
                 child_parent_map=CHILD_PARENT_CLASS_MAP, 
                 ratio_of_samples_to_use=ratio_of_samples_to_use)
   

### Combine all parsed datasets

In [ ]:
import shutil

PARSED_DATASET_LIST = [dataset_path.strip().split('/')[-1] +  '_cropped_for_DINOv2_' + str(NUM_CLASSES) + '_class' for dataset_path in DATASET_PATHS]

COMBINED_DATASET_NAME = 'down_sampled_all_' + str(NUM_CLASSES) + '_class_for_DINOv2'

if not os.path.exists(os.path.join('data', COMBINED_DATASET_NAME)):
    os.mkdir(os.path.join('data', COMBINED_DATASET_NAME))
if not os.path.exists(os.path.join('data', COMBINED_DATASET_NAME, 'images')):
    os.mkdir(os.path.join('data', COMBINED_DATASET_NAME, 'images'))
if not os.path.exists(os.path.join('data', COMBINED_DATASET_NAME, 'images', 'test')):
    os.mkdir(os.path.join('data', COMBINED_DATASET_NAME, 'images', 'test'))
if not os.path.exists(os.path.join('data', COMBINED_DATASET_NAME, 'images', 'train')):
    os.mkdir(os.path.join('data', COMBINED_DATASET_NAME, 'images', 'train'))
if not os.path.exists(os.path.join('data', COMBINED_DATASET_NAME,  'masks')):
    os.mkdir(os.path.join('data', COMBINED_DATASET_NAME, 'masks'))
if not os.path.exists(os.path.join('data', COMBINED_DATASET_NAME, 'masks', 'test')):
    os.mkdir(os.path.join('data', COMBINED_DATASET_NAME, 'masks', 'test'))
if not os.path.exists(os.path.join('data', COMBINED_DATASET_NAME, 'masks', 'train')):
    os.mkdir(os.path.join('data', COMBINED_DATASET_NAME, 'masks', 'train'))

all_images = {}
images_num = {}

for dataset_type in ['test', 'train']: 
    all_images[dataset_type] = []
    for dataset_name in PARSED_DATASET_LIST:
        all_images[dataset_type] += os.listdir(os.path.join('data', dataset_name, 'images', dataset_type))

    all_images[dataset_type] = [filename for filename in all_images[dataset_type] if 
                                filename[-3:].lower() == 'jpg' or filename[-4:].lower() == 'jpeg' ]

    images_num[dataset_type] = {}
    for filename in all_images[dataset_type] :
        if filename in images_num[dataset_type]:
            images_num[dataset_type][filename] += 1
        else:
            images_num[dataset_type][filename] = 1


for dataset_type in ['test', 'train']:
    for dataset_name in PARSED_DATASET_LIST:
        source_dir_images = os.path.join('data', dataset_name, 'images', dataset_type)
        source_dir_masks = os.path.join('data', dataset_name, 'masks', dataset_type)
        dest_dir_images = os.path.join('data', COMBINED_DATASET_NAME, 'images', dataset_type)
        dest_dir_masks = os.path.join('data', COMBINED_DATASET_NAME, 'masks', dataset_type)
    
        images_to_copy = os.listdir(source_dir_images)
        images_to_copy = [filename for filename in images_to_copy if filename[-3:].lower() == 'jpg' or filename[-4:].lower() == 'jpeg']
        for filename in images_to_copy:
            name = ".".join(filename.strip().split('.')[:-1])
            if images_num[dataset_type][filename] == 1:
                shutil.copy(os.path.join(source_dir_images, filename), os.path.join(dest_dir_images, filename))
                shutil.copy(os.path.join(source_dir_masks, name + '.pkl'), os.path.join(dest_dir_masks, name + '.pkl'))
            else:
                name_ext = '_' + str(images_num[dataset_type][filename])
                images_num[dataset_type][filename] -= 1
                shutil.copy(os.path.join(source_dir_images, filename), os.path.join(dest_dir_images, name + name_ext + '.jpg'))
                shutil.copy(os.path.join(source_dir_masks, name + '.pkl'), os.path.join(dest_dir_masks, name + name_ext + '.pkl'))